CSU Channel Island

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

url = "https://ciapps.csuci.edu/directory/Home?id=Faculty&filters=Faculty&optionsVisible=true"

response = requests.get(url)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

faculty = []

for row in soup.find_all("tr"):
    name_cell = row.find("th", scope="row")
    cells = row.find_all("td")

    if name_cell is None or len(cells) < 1:
        continue

    name_link = name_cell.find("a")

    if name_link is None:
        continue

    faculty.append({
        "name": name_link.get_text(" ", strip=True),
        "title": cells[0].get_text(" ", strip=True),
        "school": "CSU Channel Islands"
    })

# Convert the list of dictionaries to a DataFrame
df = pd.DataFrame(faculty)

# Write it to a Parquet file
df.to_parquet("csuci_faculty.parquet", index=False)

print(df.head())

                        name                                           title  \
0            Abbasi, Bahareh  Associate Professor - Mechatronics Engineering   
1              Abdolee, Reza                  Associate Professor - Comp Sci   
2        Abell, Leslie Marie          Faculty Director, Learning Communities   
3   Abramiuk, Marc Alexander                  Lecturer AY - Anthropology - 3   
4  Acosta, Christopher James             Faculty Lead - Academic Internships   

                school  
0  CSU Channel Islands  
1  CSU Channel Islands  
2  CSU Channel Islands  
3  CSU Channel Islands  
4  CSU Channel Islands  


CSU Bakersfield

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

url = "https://catalog.csub.edu/general-information/csub-information/faculty_list/"

response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

faculty = []

for row in soup.find_all("tr"):
    cells = row.find_all("td")

    # Each faculty row has: name, title
    if len(cells) != 2:
        continue

    faculty.append({
        "name": cells[0].get_text(" ", strip=True),
        "title": cells[1].get_text(" ", strip=True),
        "school": "CSU Bakersfield"
    })

df = pd.DataFrame(faculty)

print(df.head())
print("Faculty found:", len(df))

df.to_parquet("csub_faculty.parquet", index=False)

                        name  \
0         Acharya, Tathagata   
1      Acuna-Gurrola, Moises   
2  Afaqi, Mian Ahmed Shaheer   
3            Agarwal, Ankita   
4                Alali, Andy   

                                               title           school  
0     Associate Professor of Physics and Engineering  CSU Bakersfield  
1                     Assistant Professor of History  CSU Bakersfield  
2  Assistant Professor of Philosophy and Religiou...  CSU Bakersfield  
3    Assistant Professor of Management and Marketing  CSU Bakersfield  
4                        Professor of Communications  CSU Bakersfield  
Faculty found: 498


CSU Northridge


In [ ]:
!playwright install --with-deps chromium

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libasound2t64 is already the newest version (1.2.11-1ubuntu0.3).
libatk-bridge2.0-0t64 is already the newest version (2.52.0-1build1).
libatk1.

In [ ]:
!pip install playwright pyarrow
!playwright install chromium

In [ ]:
import re
import string
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

async def scrape_csun():
    faculty = []
    failed_letters = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for letter in string.ascii_lowercase:
            url = f"https://catalog.csun.edu/faculty/{letter}/"

            try:
                response = await page.goto(
                    url,
                    wait_until="domcontentloaded"
                )

                if response is None or response.status >= 400:
                    status = response.status if response else "no response"
                    print(f"Could not load {letter.upper()}: {status}")

                    failed_letters.append({
                        "letter": letter.upper(),
                        "url": url,
                        "status": status
                    })
                    continue

                soup = BeautifulSoup(
                    await page.content(),
                    "html.parser"
                )

                people_on_page = 0

                for description in soup.find_all(
                    string=re.compile(r"^\s*\(\d{4}\)")
                ):
                    bio = description.get_text(" ", strip=True)
                    name_link = description.find_previous("a")

                    if name_link is None:
                        continue

                    name = name_link.get_text(" ", strip=True)

                    title_match = re.match(
                        r"^\(\d{4}\)\s*(.*?)\.",
                        bio
                    )

                    if title_match and len(name) > 1:
                        faculty.append({
                            "name": name,
                            "title": title_match.group(1),
                            "school": "CSU Northridge",
                            "source_url": url
                        })

                        people_on_page += 1

                print(f"{letter.upper()}: {people_on_page} people found")

                # One-second pause between pages
                await page.wait_for_timeout(1000)

            except Exception as error:
                print(f"Error on {letter.upper()}: {error}")

                failed_letters.append({
                    "letter": letter.upper(),
                    "url": url,
                    "status": str(error)
                })

        await browser.close()

    return faculty, failed_letters

In [ ]:
faculty, failed_letters = await scrape_csun()

df = (
    pd.DataFrame(faculty)
    .drop_duplicates(subset=["name"])
    .sort_values("name")
)

df.to_parquet("csun_faculty.parquet", index=False)

pd.DataFrame(failed_letters).to_csv(
    "csun_failed_letters.csv",
    index=False
)

print("Faculty found:", len(df))
df.head()

A: 46 people found
B: 78 people found
C: 82 people found
D: 45 people found
E: 23 people found
F: 30 people found
G: 72 people found
H: 74 people found
I: 5 people found
J: 35 people found
K: 55 people found
L: 56 people found
M: 100 people found
N: 27 people found
O: 20 people found
P: 42 people found
Q: 6 people found
R: 59 people found
S: 95 people found
T: 39 people found
U: 1 people found
V: 24 people found
W: 47 people found
X: 1 people found
Y: 14 people found
Z: 15 people found
Faculty found: 1091


,name,title,school,source_url
0,"Abdelsayed, Michael",Assistant Professor of Biology,CSU Northridge,https://catalog.csun.edu/faculty/a/
1,"Abolghasem, Sepideh",Professor of Manufacturing Systems Engineering...,CSU Northridge,https://catalog.csun.edu/faculty/a/
2,"Abrego, Bernardo",Professor of Mathematics,CSU Northridge,https://catalog.csun.edu/faculty/a/
3,"Abrishami, Doris",Professor of Health Sciences,CSU Northridge,https://catalog.csun.edu/faculty/a/
4,"Abrol, Ravinder",Professor of Chemistry and Biochemistry,CSU Northridge,https://catalog.csun.edu/faculty/a/


CSU Los Angeles


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

faculty = []

for i in range(1,13):
  url = f"https://www.calstatela.edu/facultydirectory?page={i}"
  response = requests.get(
      url,
      headers={"User-Agent": "Mozilla/5.0"}
  )
  response.raise_for_status()

  soup = BeautifulSoup(response.text, "html.parser")



  for row in soup.find_all("tr"):
      cells = row.find_all("td")

      # Faculty rows have: name, department, email
      if len(cells) < 2:
          continue

      name = cells[0].get_text(" ", strip=True)
      department = cells[1].get_text(" ", strip=True)

      # Ignore the table header or blank rows
      if not name or name == "Name":
          continue

      faculty.append({
          "name": name,
          "department": department,
          "school": "CSU Los Angeles"
      })

df = pd.DataFrame(faculty).drop_duplicates(subset=["name"])

print(df.head())
print("Faculty found:", len(df))

df.to_parquet("csula_faculty.parquet", index=False)

                  name                                    department  \
0       Best, Sherwood  Department of Special Education & Counseling   
1         Beyer, Robbi                     Department of Kinesiology   
2       Bezdecny, Kris           Geography, Geology, and Environment   
3         Black, Sarah                            Extended Education   
4  Blaszczynski, Carol                      Department of Management   

            school  
0  CSU Los Angeles  
1  CSU Los Angeles  
2  CSU Los Angeles  
3  CSU Los Angeles  
4  CSU Los Angeles  
Faculty found: 480


CSU Long Beach


In [ ]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

urls = [
     "https://sso.csulb.edu",
 "https://www.csulb.edu/cinematic-arts/department-of-cinematic-arts-directory",
                                              "https://www.csulb.edu/school-of-art/faculty-and-staff",
                 "https://www.csulb.edu/dance/faculty-staff-directory",
             "https://web.csulb.edu/colleges/cota/music/faculty-&-staff/",
                                                     "https://www.csulb.edu/theatre-arts/faculty-staff",
          "https://www.csulb.edu/college-of-education/advanced-studies-education-and-counseling/faculty-staff",
                                  " https://www.csulb.edu/college-of-education/college-of-education-faculty-staff",
"https://www.csulb.edu/college-of-education/educational-leadership/faculty-staff",
                                        "https://www.csulb.edu/college-of-education/liberal-studies/faculty-staff",
                                                    "https://www.csulb.edu/college-of-education/teacher-education/faculty-staff-0",
                               "http://web.csulb.edu/colleges/cba/contact/index.php?dept=1",
                                      "https://web.csulb.edu/colleges/cob/contact/index.php?dept=3",
                                        "https://web.csulb.edu/colleges/cob/contact/index.php?dept=4",
                            "http://web.csulb.edu/colleges/cba/contact/index.php?dept=5",
                                   "https://web.csulb.edu/colleges/cob/contact/index.php?dept=6",
                                          "https://www.csulb.edu/hung-family-college-of-engineering/biomedical-engineering/faculty-staff",
                                "https://www.csulb.edu/hung-family-college-of-engineering/chemical-engineering/faculty-staff",
                      "https://www.csulb.edu/college-of-engineering/civil-engineering-construction-engineering-management/faculty-staff",
                         "https://www.csulb.edu/college-of-engineering/computer-engineering-computer-science/faculty-staff",
           "https://www.csulb.edu/hung-family-college-of-engineering/electrical-engineering/faculty-staff",
                             "https://www.csulb.edu/hung-family-college-of-engineering/mechanical-aerospace-engineering/mae-people",
                                            "https://www.csulb.edu/college-of-health-human-services/health-science/faculty-staff",

                                                   "https://www.csulb.edu/college-of-health-human-services/health-care-administration/faculty-staff",
                        "https://www.csulb.edu/college-of-health-human-services/school-of-social-work/faculty-staff",
                            "https://www.csulb.edu/college-of-health-human-services/kinesiology/faculty-staff-0",
                     "https://www.csulb.edu/college-of-health-human-services/school-of-nursing/faculty-and-staff",
                                                    "https://www.csulb.edu/college-of-health-human-services/speech-language-pathology/faculty-staff",
                                       "https://www.csulb.edu/college-of-health-human-services/physical-therapy/faculty-staff",
                                                  "https://www.csulb.edu/college-of-health-human-services/gerontology/faculty-staff",
                            "https://www.csulb.edu/college-of-health-human-services/child-development-family-studies/faculty",
     "https://www.csulb.edu/college-of-health-human-services/recreation-and-leisure-studies/faculty-staff",
     "https://www.csulb.edu/college-of-health-human-services/family-and-consumer-sciences/faculty-and-staff",
     "https://www.csulb.edu/college-of-health-human-services/public-policy-and-administration/faculty-staff",
     "https://www.csulb.edu/college-of-health-human-services/hospitality-management/faculty",
     "https://www.csulb.edu/college-of-education/liberal-studies/faculty-staff",
     "https://www.csulb.edu/college-of-liberal-arts/psychology/faculty-staff",
     "https://www.csulb.edu/college-of-liberal-arts/anthropology/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/philosophy/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/chicano-and-latino-studies/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/english/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/africana-studies/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/american-indian-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/religious-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/asl-linguistics-deaf-cultures-program/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/global-studies/core-faculty",
     "https://www.csulb.edu/college-of-liberal-arts/comparative-world-literature-and-classics/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/department-of-history/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/american-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/economics/department-faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/geography/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/human-development/hdev-faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/journalism-public-relations/faculty",
     "https://www.csulb.edu/school-of-art/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/asian-and-asian-american-studies/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/communication-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/environmental-science-policy/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/modern-jewish-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/linguistics-department/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/political-science/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/romance-german-russian-languages-and-literatures/people",
     "https://www.csulb.edu/college-of-liberal-arts/sociology/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/womens-gender-sexuality-studies/wgss-faculty-staff",
     "https://www.csulb.edu/biological-sciences/department-directory",
     "https://www.csulb.edu/chemistry-biochemistry/faculty",
     "https://www.csulb.edu/earth-science/department-directory",
     "https://www.csulb.edu/mathematics-statistics/department-directory",
     "https://www.csulb.edu/physics-astronomy/department-directory",
     "https://www.csulb.edu/science-education/department-directory",
     "https://www.csulb.edu/college-of-liberal-arts/environmental-science-policy/faculty"

]

import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

people = []
failed_pages = []
no_directory_table = []


def clean_column_name(column):
    """Makes column names easier to compare."""
    return re.sub(r"[^a-z]", "", str(column).lower())


def find_column(columns, options):
    """
    Finds a DataFrame column whose cleaned name contains
    one of the requested options.
    """
    for column in columns:
        cleaned = clean_column_name(column)

        for option in options:
            if option in cleaned:
                return column

    return None


for raw_url in urls:
    url = raw_url.strip()

    # This is a login page, not a faculty directory
    if "sso.csulb.edu" in url:
        continue

    try:
        response = session.get(url, timeout=20)
        response.raise_for_status()

    except requests.RequestException as error:
        print(f"Could not load: {url}")
        failed_pages.append({
            "source_url": url,
            "error": str(error)
        })
        continue

    soup = BeautifulSoup(response.text, "html.parser")
    page_people_found = 0

    # Look through every table on the current page
    for table in soup.find_all("table"):
        try:
            table_df = pd.read_html(StringIO(str(table)))[0]
        except ValueError:
            continue

        # Turns multi-level column names into regular text
        table_df.columns = [
            " ".join(map(str, col)) if isinstance(col, tuple) else str(col)
            for col in table_df.columns
        ]

        # Find columns even when their order differs from page to page
        first_name_col = find_column(table_df.columns, ["firstname"])
        last_name_col = find_column(table_df.columns, ["lastname"])
        name_col = find_column(
            table_df.columns,
            ["name", "faculty", "person", "employee"]
        )

        title_col = find_column(
            table_df.columns,
            ["title", "position", "rank", "role"]
        )

        department_col = find_column(
            table_df.columns,
            ["department", "area", "unit", "program"]
        )

        email_col = find_column(
            table_df.columns,
            ["email", "contact"]
        )

        # Skip tables that do not look like people directories
        if name_col is None and not (first_name_col and last_name_col):
            continue

        for _, row in table_df.iterrows():

            # Handles separate First Name / Last Name columns
            if first_name_col and last_name_col:
                name = (
                    f"{row[first_name_col]} {row[last_name_col]}"
                ).strip()

            else:
                name = str(row[name_col]).strip()

            # Skip empty rows and repeated table headers
            if not name or name.lower() in ["name", "nan"]:
                continue

            title = (
                str(row[title_col]).strip()
                if title_col and pd.notna(row[title_col])
                else None
            )

            department = (
                str(row[department_col]).strip()
                if department_col and pd.notna(row[department_col])
                else None
            )

            email = (
                str(row[email_col]).strip()
                if email_col and pd.notna(row[email_col])
                else None
            )

            # Removes phone numbers or extra contact text if email is mixed in
            if email:
                email_match = re.search(
                    r"[\w.\-+]+@[\w.\-]+\.\w+",
                    email
                )
                email = email_match.group(0) if email_match else None

            people.append({
                "name": name,
                "title": title,
                "department": department,
                "school": "CSU Long Beach",

            })

            page_people_found += 1

    if page_people_found == 0:
        no_directory_table.append({
            "source_url": url
        })

    print(f"{page_people_found} people found: {url}")


# Main faculty/staff output
if people:
    df = pd.DataFrame(people)

    df = (
        df.drop_duplicates(subset=["name", "department"])
          .sort_values(["department", "name"], na_position="last")
    )

    df.to_parquet("csulb_faculty.parquet", index=False)

    print("\nTotal people found:", len(df))
    print(df.head())

else:
    print("No faculty or staff tables were found.")


# Pages that failed to load
pd.DataFrame(failed_pages).to_csv(
    "csulb_failed_pages.csv",
    index=False
)

# Pages that loaded but do not have a recognizable table
pd.DataFrame(no_directory_table).to_csv(
    "csulb_pages_needing_custom_parser.csv",
    index=False
)

46 people found: https://www.csulb.edu/cinematic-arts/department-of-cinematic-arts-directory
75 people found: https://www.csulb.edu/school-of-art/faculty-and-staff
0 people found: https://www.csulb.edu/dance/faculty-staff-directory
0 people found: https://web.csulb.edu/colleges/cota/music/faculty-&-staff/
0 people found: https://www.csulb.edu/theatre-arts/faculty-staff
28 people found: https://www.csulb.edu/college-of-education/advanced-studies-education-and-counseling/faculty-staff
115 people found: https://www.csulb.edu/college-of-education/college-of-education-faculty-staff
16 people found: https://www.csulb.edu/college-of-education/educational-leadership/faculty-staff
15 people found: https://www.csulb.edu/college-of-education/liberal-studies/faculty-staff
17 people found: https://www.csulb.edu/college-of-education/teacher-education/faculty-staff-0
34 people found: http://web.csulb.edu/colleges/cba/contact/index.php?dept=1
34 people found: https://web.csulb.edu/colleges/cob/contact